In [ ]:
%pip install earthaccess cartopy

In [ ]:
from datetime import timedelta
from pathlib import Path
import shutil
import zipfile

import cartopy.crs as ccrs
import earthaccess
from earthaccess.results import DataGranule
import gdown
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio import features
from rasterio.crs import CRS
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject
from rasterio.windows import Window
from shapely.geometry import box
from sklearn.model_selection import train_test_split

In [ ]:
hwds_path = Path("hwds")
rtc_path = hwds_path / "RTC"

data_paths = {
    "RTC": hwds_path / "RTC",
    "RAW": rtc_path / "RAW",
    "WGS84": rtc_path / "WGS84",
    "MERGE": rtc_path / "MERGE",
    "CHIPS": rtc_path / "CHIPS",
    "CHIPS_TM": rtc_path / "CHIPS_TM",
    "PLOTS": rtc_path / "PLOTS",
    "SPLITS": rtc_path / "SPLITS",
}

# for p in ("CHIPS", "CHIPS_TM", "SPLITS"):
#     shutil.rmtree(data_paths[p], ignore_errors=True)

for p in data_paths.values():
    p.mkdir(parents=True, exist_ok=True)


In [ ]:
# use 60-swath version
hwds_google_drive_id = "1h_JIEcrrUF3OSTrmwAKNPa0eUEhPA2Xx"
drive_url = f"https://drive.google.com/uc?id={hwds_google_drive_id}"

shp_dir = hwds_path / "SHP"
shp_dir.mkdir(parents=True, exist_ok=True)

filename = "hwds_v3_20250205_subset_60.zip"

zip_path = shp_dir / filename

if not zip_path.exists():
    gdown.download(drive_url, str(zip_path), quiet=False)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(path=shp_dir)

shp_path = shp_dir / "hwds_v3_20250205_subset_60.shp"
gdf = gpd.read_file(shp_path)

gdf["swathDate"] = pd.to_datetime(gdf["swathDate"], format="%Y-%m-%d")
gdf["ls5hlsDate"] = pd.to_datetime(gdf["ls5hlsDate"], format="%Y-%m-%d")
gdf["s1Date"] = pd.to_datetime(gdf["s1Date"], format="%Y-%m-%d")


In [ ]:
# make some additional columns that represent buffers after projecting to UTM 15N
gdf = gdf.to_crs(32615)
buffered_event = gdf.buffer(3000)
buffered_event_background = gdf.buffer(10000)
gdf = gdf.to_crs(4326)

gdf["buffered_event"] = buffered_event
gdf["buffered_event_background"] = buffered_event_background
gdf["buffered_event"] = gdf["buffered_event"].to_crs("EPSG:4326")
gdf["buffered_event_background"] = gdf["buffered_event_background"].to_crs(
    "EPSG:4326"
)

In [ ]:
keepers = [1442, 622, 1079, 628]
gdf = gdf[gdf["swathID"].isin(keepers)]

In [ ]:
earthaccess.login()

In [ ]:
def _reproject_file(local_file: Path, reprojected_file: Path, epsg=4326) -> None:
    # https://rasterio.readthedocs.io/en/stable/topics/reproject.html#reprojecting-a-geotiff-dataset
    with rasterio.open(local_file) as src:
        dst_crs = CRS.from_epsg(epsg)
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )

        dst_kwargs = src.meta.copy()
        dst_kwargs.update(
            {"crs": dst_crs, "transform": transform, "width": width, "height": height}
        )

        with rasterio.open(reprojected_file, "w", **dst_kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                )


In [ ]:
def _make_swath_id(swathID):
    return f"{int(swathID):04d}"

In [ ]:
reprojected_swath_data = []
for _, swath in gdf.iterrows():
    swathID = _make_swath_id(swath['swathID'])

    start_date = swath["s1Date"]
    final_date = start_date + timedelta(days=1)

    date_range = (start_date.strftime("%Y-%m-%d"), final_date.strftime("%Y-%m-%d"))

    results = earthaccess.search_data(
        short_name=["OPERA_L2_RTC-S1_V1"],
        temporal=date_range,
        bounding_box=swath["geometry"].bounds,
    )

    local_files = earthaccess.download(
        results, local_path=data_paths["RAW"], show_progress=True
    )
    data_tifs = [f for f in local_files if f.name.endswith(".tif")]

    if len(data_tifs) == 0:
        print(f'No data found for swath: {swathID}')
        continue

    reprojected_paths = [
        Path(data_paths['WGS84']) / f"{swathID}.{tif.name}" for tif in data_tifs
    ]

    for granule, output_path in zip(data_tifs, reprojected_paths):
        if output_path.exists():
            continue

        print(f"reprojecting to wgs84: {output_path.name}")
        _reproject_file(granule, output_path)

    reprojected_swath_data.append(reprojected_paths)


In [ ]:
def _merge(band_files: list[Path], output_file: Path) -> Path:
    band_datasets = [rasterio.open(rtc_tif) for rtc_tif in band_files]

    try:
        mosaic, out_trans = merge(band_datasets)
        mosaic = np.squeeze(mosaic)

        out_meta = band_datasets[0].meta.copy()

        out_meta.update(
            {
                "driver": "GTiff",
                "height": mosaic.shape[0],
                "width": mosaic.shape[1],
                "transform": out_trans,
                "crs": band_datasets[0].crs,
            }
        )

        with rasterio.open(output_file, "w", **out_meta) as dst:
            dst.write(mosaic, 1)
    finally:
        for ds in band_datasets:
            ds.close()

    return output_file


In [ ]:
def _make_merged_name(template_filename: str) -> str:
    """
    https://hyp3-docs.asf.alaska.edu/guides/opera_rtc_product_guide/#naming-convention
    swathID.OPERA_L2_RTC-S1_[BurstID]_[StartDateTime]_[ProductGenerationDateTime] _[Sensor]_[PixelSpacing]_[ProductVersion]_[LayerName].Ext

    Input:   1442.OPERA_L2_RTC-S1_T063-133415-IW2_20170620T001327Z_20250925T045340Z_S1A_30_v1.0_VV.tif

    Returns: 1442.OPERA_L2_RTC-133415-IW2_20170620_S1A_30_v1.0_VV.tif
    """

    # ['1442.OPERA', 'L2', 'RTC-S1', 'T063-133415-IW2', '20170620T001327Z', '20250925T045340Z', 'S1A', '30', 'v1.0', 'VV.tif']
    name_parts = template_filename.split("_")

    name_parts.pop(5)  # Remove Product Generation Time
    name_parts.pop(3)  # Remove Burst ID

    return "_".join(name_parts)


In [ ]:
merged_swath_data = []
for reprojected_paths in reprojected_swath_data:
    merged = {}
    for band in ("VV", "VH", "mask"):
        band_files = [f for f in reprojected_paths if band in f.name]

        merged_name = _make_merged_name(band_files[0].name)
        merged_band_path = _merge(
            band_files, output_file=data_paths["MERGE"] / merged_name
        )

        merged[band] = merged_band_path

    merged_swath_data.append(merged)


In [ ]:
def _rename(path: Path, extension: str, mask_name: str) -> Path:
    return path.parent / path.name.replace(extension, mask_name)

In [ ]:
def _generate_masks(
    merged_path: Path, swath: pd.Series, merged_extension: str
) -> tuple[Path, Path]:

    event_path = _rename(merged_path, merged_extension, "EVENT.tif")
    mask_path = _rename(merged_path, merged_extension, "MASK.tif")

    with rasterio.open(merged_path) as ds:
        profile = ds.profile

        profile.update(
            tiled=True,
            blockxsize=512,
            blockysize=512
        )

        mask_raster = features.rasterize(
            shapes=[[swath["geometry"], 1]],
            fill=0,
            out_shape=ds.shape,
            transform=ds.transform,
        )

        with rasterio.open(mask_path, "w", **profile) as dst:
            dst.write(mask_raster, 1)
            print("generated:", mask_path)

        event_mask = features.rasterize(
            shapes=[
                [swath["buffered_event_background"], 3],
                [swath["buffered_event"], 2],
                [swath["geometry"], 1],
            ],
            fill=0,
            out_shape=ds.shape,
            transform=ds.transform,
        )

        with rasterio.open(event_path, "w", **profile) as dst:
            dst.write(event_mask, 1)
            print("generated:", event_path)

    return event_path, mask_path

In [ ]:
merged_swath_data_with_masks = []

for merged in merged_swath_data:
    swath_id = int(merged['VV'].name[:4])
    swath = gdf[gdf['swathID'] == swath_id].iloc[0]

    event_tif, mask_tif = _generate_masks(
        merged["VV"], swath, merged_extension="VV.tif"
    )

    merged_swath_data_with_masks.append({
        **merged,
        "EVENT": event_tif,
        "MASK": mask_tif,
    })


In [ ]:
MINIMUM_VALID_DATA_PERCENT = 50.0

def _is_valid_rtc(merged: dict[str, Path]) -> bool:
    with rasterio.open(merged["mask"]) as ds:
        validity_mask = ds.read(1)

    with rasterio.open(merged["EVENT"]) as ds:
        event_mask = ds.read(1)

    is_event_pixel = event_mask == 1
    # https://hyp3-docs.asf.alaska.edu/guides/opera_rtc_product_guide/#validity-mask
    is_valid_pixel = np.isin(validity_mask, [0, 1])

    total_event_pixels = is_event_pixel.sum()
    valid_event_pixels = (is_event_pixel & is_valid_pixel).sum()

    pct_valid_data = 100.0 * valid_event_pixels / total_event_pixels
    print(f"Percent of the event with valid data: {pct_valid_data:.1f}%")

    return pct_valid_data > MINIMUM_VALID_DATA_PERCENT

In [ ]:
valid_rtc_data_swaths = []
for merged_data in merged_swath_data_with_masks:

    if merged_data is None or not _is_valid_rtc(merged_data):
        print("Skipping: not enough valid data")
        continue

    valid_rtc_data_swaths.append(merged_data)


In [ ]:
def _stack_rtc_bands(merged: dict[str, Path], data_bands: tuple[str]) -> None:
    with rasterio.open(merged[data_bands[0]]) as src:
        meta = src.meta.copy()

    meta.update(count=len(data_bands), dtype=np.float32)
    stacked_file_name = _rename(merged["VV"], "VV.tif", "BANDS.tif")

    with rasterio.open(stacked_file_name, "w", **meta) as dst:
        for idx, band in enumerate(data_bands, start=1):
            with rasterio.open(merged[band]) as src:
                dst.write(src.read(1), idx)

    merged["BANDS"] = stacked_file_name

    return merged


In [ ]:
stacked_swath_data = []
for merged_rtc_data in valid_rtc_data_swaths:
    stacked_data = _stack_rtc_bands(merged_rtc_data, data_bands=("VV", "VH"))
    stacked_swath_data.append(stacked_data)


In [ ]:
def _chip_rtc_data(merged: dict[str, Path], data_paths: dict[str, Path], chip_size=256):
    chips = {}

    grid = []
    with rasterio.open(merged["BANDS"]) as ref:
        n_cols = ref.width // chip_size
        n_rows = ref.height // chip_size

        for row in range(n_rows):
            for col in range(n_cols):
                window = Window(col * chip_size, row * chip_size, chip_size, chip_size)
                bounds = ref.window_bounds(window)

                tile_id = f"{row:03d}.{col:03d}"
                chips[tile_id] = {}
                grid.append((tile_id, bounds))

    for chip_layer in ("BANDS", "EVENT", "MASK"):
        layer_path = merged[chip_layer]

        with rasterio.open(layer_path) as src:
            for tile_id, bounds in grid:
                window = src.window(*bounds)
                window = Window(
                    round(window.col_off),
                    round(window.row_off),
                    round(window.width),
                    round(window.height),
                )

                data = src.read(window=window)

                chip_meta = src.meta.copy()
                chip_meta.update(
                    {
                        "width": window.width,
                        "height": window.height,
                        "transform": src.window_transform(window),
                    }
                )

                chip_name = layer_path.name.replace(
                    f"{chip_layer}.tif", f"{tile_id}.{chip_layer}.tif"
                )
                chip_path = data_paths["CHIPS"] / chip_name

                with rasterio.open(chip_path, "w", **chip_meta) as dst:
                    dst.write(data)

                chips[tile_id][chip_layer] = chip_path

    return chips


In [ ]:
chips_for_swaths = []
for stacked_data in stacked_swath_data:
    print(f'chipping swath: {stacked_data['BANDS'].name[:4]}' )
    chips = _chip_rtc_data(stacked_data, data_paths)
    chips_for_swaths.append(chips)


In [ ]:
def _filter_chips(chips: dict[str, dict]) -> list[dict]:
    good_chips = []

    for tile_id, chip in chips.items():
        with rasterio.open(chip["BANDS"]) as ds:
            rtc_data = ds.read()

        with rasterio.open(chip["EVENT"]) as ds:
            event_mask = ds.read(1)

        has_nan_pixels = np.isnan(rtc_data).sum() > 0

        num_pixels = event_mask.size
        num_event_pixels = np.count_nonzero(event_mask > 0)

        pct_pixels_over_event = 100.0 * (num_event_pixels / num_pixels)
        data_overlaps_event = pct_pixels_over_event > 1

        if not has_nan_pixels and data_overlaps_event:
            good_chips.append(chip)

    return good_chips


In [ ]:
good_swath_chips = []
for chips in chips_for_swaths:
    good_chips = _filter_chips(chips)
    good_swath_chips.append(good_chips)


In [ ]:
for good_chips in good_swath_chips:
    print(f"Found {len(good_chips)} good chips")
    for chip in good_chips:
        for band, chip_path in chip.items():
            if band not in ("MASK", "BANDS"):
                continue

            dest = data_paths["CHIPS_TM"] / chip_path.name
            shutil.copy(chip_path, dest)


In [ ]:
def _plot_chips(
    merged_band_file,
    all_chips,
    good_chips,
    swath,
    save_to: Path | None = None,
    quite=False,
):
    crs_pc = ccrs.PlateCarree()

    with rasterio.open(merged_band_file) as ds:
        bounds = ds.bounds
        full_extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
        rtc_data = ds.read()

    def normalize_image_array(
        input_array: np.ndarray, vmin: float, vmax: float
    ) -> np.ndarray:
        input_array = input_array.astype(float)
        scaled_array = (input_array - vmin) / (vmax - vmin)
        scaled_array[np.isnan(input_array)] = 0
        normalized_array = np.round(np.clip(scaled_array, 0, 1) * 255).astype(np.uint8)

        return normalized_array

    vv = normalize_image_array(np.sqrt(rtc_data[0]), 0.14, 0.52)
    vh = normalize_image_array(np.sqrt(rtc_data[1]), 0.05, 0.259)
    img = np.stack([vv, vh, vv], axis=-1)

    # plot BANDS and geom
    fig, ax = plt.subplots(
        1,
        1,
        subplot_kw={"projection": crs_pc},
        figsize=(12, 12),
        layout="constrained",
    )

    swath_geom = swath["geometry"]

    ax.imshow(img, extent=full_extent, origin="upper", transform=crs_pc)
    ax.add_geometries(
        [swath_geom], edgecolor="red", linewidth=2, facecolor="none", crs=crs_pc
    )

    def show_chips(chips, color, linewidth, z):
        for chip in chips:
            with rasterio.open(chip) as ds:
                chip_bounds = ds.bounds
                chip_geom = box(
                    chip_bounds.left,
                    chip_bounds.bottom,
                    chip_bounds.right,
                    chip_bounds.top,
                )

            ax.add_geometries(
                [chip_geom],
                edgecolor=color,
                linewidth=linewidth,
                alpha=1,
                zorder=z,
                facecolor="none",
                crs=crs_pc,
            )

    show_chips(all_chips, "yellow", 1, z=1)
    show_chips(good_chips, "blue", 3, z=2)

    ax.set_extent(full_extent, crs=crs_pc)

    if save_to:
        plt.savefig(
            save_to / f"{merged_band_file.name.removesuffix('BANDS.tif')}.png",
            dpi=300,
            bbox_inches="tight",
        )

    if not quite:
        plt.show()

    plt.close(fig)


In [ ]:
for _, swath in gdf.iterrows():
    swath_id = _make_swath_id(swath["swathID"])

    merged_file = list(data_paths["MERGE"].glob(f"{swath_id}.*BANDS.tif"))
    if len(merged_file) == 0:
        print(f"no chips for {swath_id}")
        continue

    all_chips = list(data_paths["CHIPS"].glob(f"{swath_id}.*.tif"))
    good_chips = list(data_paths["CHIPS_TM"].glob(f"{swath_id}.*.tif"))

    print(f"plotting {swath_id}")

    _plot_chips(
        merged_file[0], all_chips, good_chips, swath, save_to=data_paths["PLOTS"]
    )


In [ ]:
band_chips = list(data_paths["CHIPS_TM"].glob("*.BANDS.tif"))

In [ ]:
chip_ids = [p.name.removesuffix(".BANDS.tif") for p in band_chips]

train, test = train_test_split(chip_ids, test_size=0.3, random_state=42)

splits = {"train": train, "val": test, "test": test}

for split, chip_ids in splits.items():
    split_path = data_paths['SPLITS'] / f"{split}.txt"
    split_path.write_text("\n".join(chip_ids))


In [ ]:
n_bands = 2 # VV, VH
mean = np.zeros(n_bands, dtype=np.float64)
M2 = np.zeros(n_bands, dtype=np.float64)
count = np.zeros(n_bands, dtype=np.float64)

for chip in band_chips:
    with rasterio.open(chip) as src:
        band_data = src.read()
        count, mean, M2 = 0, 0, 0

        _, H, W = band_data.shape

        batch_count = H * W
        batch_mean = band_data.mean(axis=(1, 2))
        batch_var = band_data.var(axis=(1, 2))

        delta = batch_mean - mean
        total_count = count + batch_count

        mean = mean + delta * (batch_count / total_count)
        M2 = (
            M2
            + batch_var * batch_count
            + (delta**2) * count * batch_count / total_count
        )
        count = total_count

variance = M2 / count
std = np.sqrt(variance)


In [ ]:
print(f"Means (VV, VH): {mean}")
print(f"Stds (VV, VH): {std}")
(hwds_path / 'stats.txt').write_text(
    f"Means (VV, VH): {mean}\n"
    f"Stds (VV, VH): {std}"
)

In [ ]:
!mkdir hwds-rtc

!cp hwds/stats.txt hwds-rtc
!cp hwds/RTC/CHIPS_TM hwds-rtc -r
!cp hwds/RTC/SPLITS hwds-rtc -r
!cp hwds/RTC/PLOTS hwds-rtc -r


In [ ]:
!zip "hwds-rtc.zip" hwds-rtc -r